
**Business Case: AI Customer Complaint Triage Assistant**
Imagine an e-commerce company receives 5,000–10,000 customer complaints every day through email, chat and support tickets.

Examples:

"My order was supposed to arrive yesterday. Nobody is responding and I need it urgently."
"₹4,999 was deducted from my card but the order failed."
"The laptop I received has a broken screen. I want a replacement."

Today, support executives manually read each ticket and decide:
What is the problem? → How serious is it? → Which team should handle it? → What information should we capture? → What should we tell the customer?
Our LLM will automate the first-level analysis, not make irreversible business decisions.

| Stage | NLP task               | Business output                               |
| ----- | ---------------------- | --------------------------------------------- |
| 1     | Sentiment analysis     | Positive / Neutral / Negative                 |
| 2     | Classification         | Delivery / Refund / Payment / Product / Other |
| 3     | Information extraction | Order ID, product, amount, issue              |
| 4     | Summarization          | Short agent summary                           |
| 5     | Response generation    | Draft customer reply                          |
| 6     | Structured output      | JSON for CRM/helpdesk integration             |


CUSTOMER COMPLAINT <br>
        ↓ <br>
      LLM <br>
        ↓ <br>

 │ Sentiment           │ <br>
 │ Classification      │ <br>
 │ Information Extract │ <br>
 │ Summarization       │ <br>
 │ Response Generation │ <br>
        ↓ <br>
 STRUCTURED JSON <br>
        ↓ <br>
CRM / SUPPORT SYSTEM <br>
        ↓ <br>
Customer Support Agent <br>

In [13]:
#Part 1 — Setup in Google Colab
#We can use a small Hugging Face instruction model so we don't need an API key.

!pip install -q -U transformers accelerate torch
#It silently installs the Python libraries needed to load, run, and tokenize Hugging Face Transformer/LLM models efficiently



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2

In [14]:
#Load Qwen
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Qwen loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully!


What happened?

pipeline() gives us a simple Hugging Face interface.

Instruction-tuned model. It is designed to respond to instructions such as:

Classify this complaint...
Summarize this text...
Extract information...

So instead of training a model, we're going to instruct an existing model.

In [15]:
# 3. Create a reusable Qwen function
def ask_qwen(prompt, max_new_tokens=200):

    messages = [
        {
            "role": "system",
            "content": "You are a helpful business AI assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids
        in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return response

In [16]:
print(
    ask_qwen(
        "Explain artificial intelligence in one sentence."
    )
)

Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems, such as learning, reasoning, and self-correction.


In [23]:
#4. Business Case: Customer Complaint Triage
#Create one customer ticket.
ticket = """
Customer Name: Rajesh Kumar
Order ID: ORD78291

I ordered an iPhone 17 for Rs. 82,000.

It was supposed to reach Hyderabad yesterday,
but I still haven't received it.

The tracking page hasn't updated for two days.

I need the phone urgently for a business trip.

Please resolve this immediately.
"""

#5. NLP Task 1 — Sentiment Analysis
#We are doing zero-shot classification. We did not train Qwen. We simply told it: Task + Possible labels + Input

prompt = f"""
Analyze the sentiment of the following customer complaint.

Choose exactly one:

Positive
Neutral
Negative

Customer complaint:

{ticket}

Return only the sentiment label.
"""

response = ask_qwen(prompt)

print("Category:",response)


#6. NLP Task 2 — Complaint Classification
prompt = f"""
Classify the following customer complaint.

Choose exactly one category:

Delivery
Refund
Payment
Product Defect
Cancellation
Other

Customer complaint:

{ticket}

Return only the category.
"""

response = ask_qwen(prompt)

print("Sentiment:",response)

#7. Few-Shot Classification (Examples)
prompt = f"""
You are a customer complaint classification system.

Examples:

Complaint:
My credit card was charged but the order failed.

Category:
Payment


Complaint:
The shoes arrived damaged.

Category:
Product Defect


Complaint:
I cancelled my order but still have not received my money.

Category:
Refund


Complaint:
My package was supposed to arrive yesterday but it has not arrived.

Category:
Delivery


Now classify the following complaint:

{ticket}

Choose from:

Delivery
Refund
Payment
Product Defect
Cancellation
Other

Return only the category.
"""

response = ask_qwen(prompt)

print("Few shot category:",response)

#8. NLP Task 3 — Information Extraction
#Now extract business information.

prompt = f"""
Extract information from the customer complaint.

Return the following fields:

Customer Name:
Order ID:
Product:
Amount:
Location:
Problem:

If information is unavailable,
write "Not Available".

Customer complaint:

{ticket}
"""

response = ask_qwen(prompt)

print("Information extraction:\n",response)

#9. NLP Task 4 — Summarization
prompt = f"""
You are assisting a customer-support executive.

Summarize the following customer complaint
in maximum 30 words.

Include:

1. Main problem
2. Product
3. Order ID
4. Customer urgency

Customer complaint:

{ticket}

Return only the summary.
"""

response = ask_qwen(
    prompt,
    max_new_tokens=80
)

print("Summarization:",response)

#10. NLP Task 5 — Response Generation

Category: Negative
Sentiment: Cancellation
Few shot category: Delivery
Information extraction:
 - Customer Name: Rajesh Kumar
- Order ID: ORD78291
- Product: iPhone 17
- Amount: Not Available (as no amount mentioned)
- Location: Hyderabad
- Problem: Phone not delivered as per order and tracking updates are not updating.
Summarization: Rajesh Kumar ordered an iPhone 17 for Rs. 82,000 but it's not delivered yet. He needs it urgently for a business trip and has been waiting for two days without updates on his order.


In [24]:
#10. NLP Task 5 — Response Generation
prompt = f"""
You are a professional customer-support executive
working for an e-commerce company.

Respond to the customer complaint below.

Requirements:

- Acknowledge the problem
- Be empathetic
- Mention the Order ID
- Mention that the delivery issue is being investigated
- Do not promise a specific delivery date
- Maintain a professional tone
- Maximum 100 words

Customer complaint:

{ticket}

Write the customer response.
"""

response = ask_qwen(
    prompt,
    max_new_tokens=180
)

print("Response Generation:\n",response)

Response Generation:
 Dear Mr. Kumar,

Thank you for reaching out with your concern regarding your order. We apologize for any inconvenience caused and understand how urgent this situation is for you.

Your order has been flagged as pending and we are currently investigating the delivery issue. Please rest assured that our team is working diligently to resolve this matter promptly.

We appreciate your patience during this time and will keep you informed of any updates on the status of your shipment.

Best regards,
[Your Company] Customer Support Team


In [25]:
#11. NLP Task 6 — Structured JSON Output
prompt = f"""
Analyze the following customer complaint.

Return ONLY valid JSON.

Use exactly this structure:

{{
    "customer_name": "",
    "order_id": "",
    "product": "",
    "amount": "",
    "location": "",
    "category": "",
    "sentiment": "",
    "priority": "",
    "summary": ""
}}

Category must be one of:

Delivery
Refund
Payment
Product Defect
Cancellation
Other

Sentiment must be one of:

Positive
Neutral
Negative

Priority must be one of:

Low
Medium
High

Customer complaint:

{ticket}

Return JSON only.
"""

response = ask_qwen(
    prompt,
    max_new_tokens=250
)

print(response)

```json
{
    "customer_name": "Rajesh Kumar",
    "order_id": "ORD78291",
    "product": "iPhone 17",
    "amount": "Rs. 82,000",
    "location": "Hyderabad",
    "category": "Delivery",
    "sentiment": "Negative",
    "priority": "High",
    "summary": "Rajesh Kumar ordered an iPhone 17 for Rs. 82,000 and has not received it despite delivery being scheduled for yesterday. The tracking page is not updating, causing urgency for a business trip."
}
```


In [26]:
#12. Convert the JSON response into Python
import json

prompt = f"""
Analyze the following customer complaint.

Return ONLY valid JSON.

{{
    "customer_name": "",
    "order_id": "",
    "category": "",
    "sentiment": "",
    "priority": ""
}}

Customer complaint:

{ticket}
"""

response = ask_qwen(
    prompt,
    max_new_tokens=150
)

print("Raw LLM Output:")
print(response)
import json

prompt = f"""
Analyze the following customer complaint.

Return ONLY valid JSON.

{{
    "customer_name": "",
    "order_id": "",
    "category": "",
    "sentiment": "",
    "priority": ""
}}

Customer complaint:

{ticket}
"""

response = ask_qwen(
    prompt,
    max_new_tokens=150
)

print("Raw LLM Output:")
print(response)

Raw LLM Output:
```json
{
    "customer_name": "Rajesh Kumar",
    "order_id": "ORD78291",
    "category": "Electronics",
    "sentiment": "Negative",
    "priority": "High"
}
```
Raw LLM Output:
```json
{
    "customer_name": "Rajesh Kumar",
    "order_id": "ORD78291",
    "category": "Electronics",
    "sentiment": "Negative",
    "priority": "High"
}
```


In [27]:
#13. Bad Prompt vs Good Prompt
bad_prompt = f"""
Analyze this:

{ticket}
"""

print(
    ask_qwen(
        bad_prompt,
        max_new_tokens=150
    )
)

good_prompt = f"""
You are a customer-support ticket analyst.

Analyze the following complaint.

Determine:

1. Complaint category
2. Sentiment
3. Priority
4. Main issue
5. Recommended support team

Complaint categories:

Delivery
Refund
Payment
Product Defect
Cancellation
Other

Priority:

Low
Medium
High

Return the result as a concise table.

Customer complaint:

{ticket}
"""

print(
    ask_qwen(
        good_prompt,
        max_new_tokens=200
    )
)

Based on the information provided in your message:

- **Customer Name:** Rajesh Kumar
- **Order ID:** ORD78291
- **Product Ordered:** iPhone 17
- **Price:** Rs. 82,000
- **Expected Delivery Date:** Yesterday (based on the order being placed)
- **Current Status:** Tracking page has not been updated for two days
- **Issue:** Customer needs the phone urgently for a business trip

### Analysis:
1. **Order Fulfillment Delay**: The product is expected to be delivered by yesterday but has not arrived yet.
2. **Tracking Issue**: The tracking page has not been updated for two days, indicating there might be some delay or issue
| Complains Category | Sentiment | Priority | Main Issue | Recommended Support Team |
|---|---|---|---|---|
| Delivery | Negative | High | The order has not been delivered and the tracking information is outdated. | Customer Service |


In [28]:
#14. Build One Complete AI Ticket Analyzer
def analyze_customer_ticket(ticket):

    prompt = f"""
You are an AI customer-support ticket analyst.

Analyze the following customer complaint.

Perform these tasks:

1. Identify complaint category
2. Determine sentiment
3. Determine priority
4. Identify main issue
5. Recommend the responsible team
6. Generate a short summary

Allowed categories:

Delivery
Refund
Payment
Product Defect
Cancellation
Other

Allowed sentiment:

Positive
Neutral
Negative

Allowed priority:

Low
Medium
High

Return ONLY valid JSON in the following format:

{{
    "category": "",
    "sentiment": "",
    "priority": "",
    "main_issue": "",
    "recommended_team": "",
    "summary": ""
}}

Customer complaint:

{ticket}
"""

    response = ask_qwen(
        prompt,
        max_new_tokens=250
    )

    return response


result = analyze_customer_ticket(ticket)

print(result)

```json
{
    "category": "Delivery",
    "sentiment": "Negative",
    "priority": "High",
    "main_issue": "Delayed delivery of the product",
    "recommended_team": "Shipping Team",
    "summary": "Rajesh Kumar ordered an iPhone 17 and is experiencing delays with its delivery. He needs the phone urgently for his business trip."
}
```


In [30]:
#15. Test Different Customer Complaints
#Case 1 — Payment issue
ticket1 = """
My credit card has been charged Rs. 12,500
but the order was never created.

Please refund my money immediately.
"""

print(
    analyze_customer_ticket(ticket1)
)

#Case 2 — Product defect
ticket2 = """
The headphones I received today
are not working.

The left speaker has no sound.

I want a replacement.
"""

print(
    analyze_customer_ticket(ticket2)
)

#Case 3 — Delivery issue
ticket3 = """
My package was supposed to arrive three days ago.

Tracking still shows dispatched
and there has been no update.
"""

print(
    analyze_customer_ticket(ticket3)
)

```json
{
    "category": "Payment",
    "sentiment": "Negative",
    "priority": "High",
    "main_issue": "Credit Card Charge and Order Not Created",
    "recommended_team": "Billing Team",
    "summary": "The customer is requesting immediate refund for a charge on their credit card that they believe should not have occurred due to an uncreated order."
}
```
```json
{
    "category": "Product Defect",
    "sentiment": "Negative",
    "priority": "High",
    "main_issue": "Inability to use the product due to non-functional components",
    "recommended_team": "Technical Support Team",
    "summary": "Customer reports that their headphones do not work, specifically mentioning issues with the left speaker having no sound. The customer wants a replacement."
}
```
```json
{
    "category": "Delivery",
    "sentiment": "Negative",
    "priority": "High",
    "main_issue": "Package not delivered as expected and lack of tracking updates",
    "recommended_team": "Shipping Team",
    "summary

In [31]:
#16. Analyze Multiple Tickets
tickets = [

"""
My order was supposed to arrive yesterday,
but it is still not delivered.
""",

"""
My credit card has been charged twice
for the same order.
""",

"""
The laptop I received has a cracked screen.
""",

"""
I cancelled my order five days ago
but haven't received my refund.
"""
]

#Process them
for i, customer_ticket in enumerate(tickets, start=1):

    print("=" * 50)
    print("TICKET:", i)

    result = analyze_customer_ticket(
        customer_ticket
    )

    print(result)

TICKET: 1
```json
{
    "category": "Delivery",
    "sentiment": "Negative",
    "priority": "High",
    "main_issue": "Order Delivery Delay",
    "recommended_team": "Shipping Team",
    "summary": "The customer's order has been delayed and is still not delivered, causing inconvenience."
}
```
TICKET: 2
```json
{
    "category": "Payment",
    "sentiment": "Negative",
    "priority": "High",
    "main_issue": "Credit Card Charge Error",
    "recommended_team": "Billing Team",
    "summary": "The customer is experiencing a negative payment error where their credit card has been charged twice for the same order."
}
```
TICKET: 3
```json
{
    "category": "Product Defect",
    "sentiment": "Negative",
    "priority": "High",
    "main_issue": "Cracked Screen",
    "recommended_team": "Tech Support Team",
    "summary": "The customer is reporting that their laptop, which they purchased, has a cracked screen."
}
```
TICKET: 4
```json
{
    "category": "Cancellation",
    "sentiment": "Nega

In [32]:
#17. Add Temperature for Prompt Experiments

def ask_qwen_creative(
    prompt,
    max_new_tokens=200,
    temperature=0.7
):

    messages = [
        {
            "role": "system",
            "content": "You are a helpful business AI assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids
        in zip(
            model_inputs.input_ids,
            generated_ids
        )
    ]

    response = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return response

    #Compare 1
    print(
    ask_qwen_creative(
        prompt,
        temperature=0.9
    )
)

#option 2 compare
print(
    ask_qwen_creative(
        prompt,
        temperature=0.9
    )
)

```json
{
    "customer_name": "Rajesh Kumar",
    "order_id": "ORD78291",
    "category": "Product Delivery",
    "sentiment": "Urgent and Satisfied",
    "priority": "High"
}
```


Prompt Engineering <br>
      ↓ <br>
Works reasonably well <br>
      ↓ <br>
But output may vary <br>
      ↓ <br>
We want company-specific behaviour <br>
      ↓ <br>
PEFT <br>
      ↓ <br>
LoRA / QLoRA <br>
      ↓ <br>
Fine-tuned Qwen <br>